# H-Potential: real-world data pipeline
This notebook implements the thesis-prescribed outcome-blind measurement and paired prognostic comparison. Synthetic mode is a software demonstration only. It is never empirical evidence.

In [ ]:
import os, sys, subprocess
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not Path('pyproject.toml').exists():
    raise RuntimeError('Upload/unzip the repository or clone your Git repository before continuing.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'])

## Select data mode
For a governed run set `USE_SYNTHETIC=False` and point `DATA_PATH` to the approved de-identified CSV/Parquet. Do not upload direct identifiers.

In [ ]:
USE_SYNTHETIC = True
DATA_PATH = Path('data/sample/synthetic.csv')
CONFIG_PATH = Path('configs/study.yaml')
OUTPUT_DIR = Path('outputs/colab_run')
if not USE_SYNTHETIC and not DATA_PATH.exists():
    raise FileNotFoundError('Approved real-world extract not found. Supply it through the governed route.')

In [ ]:
from hpotential.config import load_config
from hpotential.data import synthetic_dataset, load_and_validate
cfg = load_config(CONFIG_PATH)
if USE_SYNTHETIC:
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    synthetic_dataset(cfg, n=750, seed=cfg.seed).to_csv(DATA_PATH, index=False)
df, manifest = load_and_validate(DATA_PATH, cfg)
print({k: manifest[k] for k in ('rows','columns','sha256')})

## Run the frozen comparison
The measurement model is fitted without the outcome columns and saved before prognosis.

In [ ]:
from hpotential.pipeline import run
performance = run(DATA_PATH, cfg, OUTPUT_DIR)
performance

In [ ]:
import pandas as pd
equity = pd.read_csv(OUTPUT_DIR / 'equity_performance.csv')
equity.head(20)

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pytest', '-q'])
print('All software checks passed. This does not by itself validate the scientific construct.')

## Interpretation boundary
A same-cohort split is internal development evidence, not external validation. Confirmatory work must add full-pipeline paired bootstrap, multiple imputation, IPCW survival metrics, survey-design variance, competing-risk analysis, scalar invariance and independent-cohort validation as specified in `docs/PROTOCOL_AND_GAPS.md`.